##Drive e Github

In [1]:
# collego colab al drive
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [2]:
#Clonare la repository GitHub con il codice
!git clone https://github.com/GiammaBigFishEngineer/MaskArchitectureAnomaly_CourseProject.git

#!git clone https://github.com/letiBri/MaskArchitectureAnomaly_CourseProject.git

#entrare dentro la cartella "eomt" della repository
%cd MaskArchitectureAnomaly_CourseProject/eomt

import sys
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject')
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject/eomt')

Cloning into 'MaskArchitectureAnomaly_CourseProject'...
remote: Enumerating objects: 210, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 210 (delta 1), reused 0 (delta 0), pack-reused 201 (from 2)
Receiving objects: 100% (210/210), 30.42 MiB | 25.92 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/MaskArchitectureAnomaly_CourseProject/eomt


In [1]:

import sys
# First, try to uninstall numpy and ood_metrics to ensure a clean slate
!pip uninstall -y numpy ood_metrics

# Then, install ood_metrics which will install a compatible numpy version
!pip install ood_metrics



Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 102.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which 

In [3]:
# Re-import necessary modules after potential reinstallation
from ood_metrics import fpr_at_95_tpr, calc_metrics, plot_roc, plot_pr,plot_barcode
from sklearn.metrics import roc_auc_score, roc_curve, auc, precision_recall_curve, average_precision_score

In [4]:
import os
import sys
import random
import numpy as np
import torch

# Path della repository clonata in Colab
base_path = "/content/MaskArchitectureAnomaly_CourseProject"

# Aggiungo la repo e la cartella eomt al path Python
sys.path.insert(0, base_path)
sys.path.insert(0, os.path.join(base_path, "eomt"))

print("Base path exists:", os.path.exists(base_path))
print("EoMT path exists:", os.path.exists(os.path.join(base_path, "eomt")))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

NUM_CLASSES = 19

Base path exists: True
EoMT path exists: True
Device: cuda


In [5]:
!pip install lightning ood-metrics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 60.1 MB/s eta 0:00:00


In [6]:
import yaml
import importlib
import warnings
import torch

# ==============================================================================
# SELEZIONE MODELLO: Imposta True se vuoi usare COCO, False per Cityscapes
# ==============================================================================
USE_COCO = False

warnings.filterwarnings("ignore")

if USE_COCO:
    print("--- CONFIGURAZIONE MODELLO COCO ---")
    CONFIG_PATH = "configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"
    STATE_DICT_PATH = "/content/drive/MyDrive/validation_Cityscapes/eomt_coco.bin"
    MODEL_NAME = "coco"
    num_classes_to_load = 133
    init_img_size = (640, 640)  # La geometria dei pos_embed di COCO è basata su 640
else:
    print("--- CONFIGURAZIONE MODELLO CITYSCAPES ---")
    CONFIG_PATH = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
    STATE_DICT_PATH = "/content/drive/MyDrive/validation_Cityscapes/eomt_cityscapes.bin"
    MODEL_NAME = "cityscapes"
    num_classes_to_load = 19
    init_img_size = (1024, 1024)

print("1. Caricamento configurazione YAML...")
with open(CONFIG_PATH, "r") as f:
    current_config = yaml.safe_load(f)

print("2. Inizializzazione dinamica dell'Encoder (ViT/DINOv2)...")
encoder_cfg = current_config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=init_img_size, **encoder_cfg.get("init_args", {}))

print("3. Inizializzazione dinamica della Network...")
network_cfg = current_config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}
network = network_cls(
    masked_attn_enabled=False,
    num_classes=num_classes_to_load,
    encoder=encoder,
    **network_kwargs,
)

print("4. Inizializzazione del modulo PyTorch Lightning...")
lit_module_name, lit_class_name = current_config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
model_kwargs = {k: v for k, v in current_config["model"]["init_args"].items() if k != "network"}

# --- FIX COCO: Estraiamo dinamicamente le 'stuff_classes' richieste dal costruttore Panoptic ---
if "data" in current_config and "init_args" in current_config["data"] and "stuff_classes" in current_config["data"]["init_args"]:
    model_kwargs["stuff_classes"] = current_config["data"]["init_args"]["stuff_classes"]

model_kwargs_final = {k: v for k, v in model_kwargs.items()}
if "num_classes" in model_kwargs_final:
    del model_kwargs_final["num_classes"]

model = (
    lit_cls(
        img_size=init_img_size,
        num_classes=num_classes_to_load,
        network=network,
        **model_kwargs_final,
    )
    .eval()
    .to(device)
)

print("5. Caricamento dei pesi preallenati (state_dict)...")
state_dict = torch.load(STATE_DICT_PATH, map_location=device, weights_only=True)
model.load_state_dict(state_dict, strict=False)

# ----------------------------------------------------------------------
# ADATTAMENTO GEOMETRICO OPERATIVO PER L'INFERENZA A FINESTRE
# ----------------------------------------------------------------------
if USE_COCO:
    model.img_size = (640, 640)
    model.window_size = 640
    target_img_size = (640, 640)
else:
    model.img_size = (1024, 1024)
    model.window_size = 1024
    target_img_size = (1024, 1024)

print(f"\n✅ Modello EoMT [{MODEL_NAME.upper()}] caricato e configurato con successo!")

--- CONFIGURAZIONE MODELLO CITYSCAPES ---
1. Caricamento configurazione YAML...
2. Inizializzazione dinamica dell'Encoder (ViT/DINOv2)...


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

3. Inizializzazione dinamica della Network...
4. Inizializzazione del modulo PyTorch Lightning...
5. Caricamento dei pesi preallenati (state_dict)...

✅ Modello EoMT [CITYSCAPES] caricato e configurato con successo!


In [7]:
from torchvision.transforms import Compose, Resize, ToTensor
from PIL import Image

target_img_size = (512, 1024)
input_transform = Compose([
    Resize(target_img_size, interpolation=Image.BILINEAR),
    #ToTensor(),
])

target_transform = Compose([
    Resize(target_img_size, interpolation=Image.NEAREST),
])

In [8]:
import torch.nn.functional as F
from torch.amp import autocast

def get_eomt_logits(model, img_tensor, target_size):
    """
    Passa un'immagine in EoMT usando la logica a finestre/crop
    e restituisce logits/scores semantici pixel-per-pixel.

    img_tensor: [3, H, W]
    output: lista di tensori [C, H, W]
    """

    model.eval()
    model.window_size = target_size[0]

    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):

        imgs = [img_tensor.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]

        crops, origins = model.window_imgs_semantic(imgs)

        mask_logits_per_layer, class_logits_per_layer = model(crops)

        # IMPORTANTE:
        # le mask devono essere riportate alla dimensione dei crop,
        # non alla dimensione dell'immagine intera.
        crop_size = crops.shape[-2:]

        mask_logits = F.interpolate(
            mask_logits_per_layer[-1],
            size=crop_size,
            mode="bilinear",
            align_corners=False
        )

        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits,
            class_logits_per_layer[-1]
        )

        logits = model.revert_window_logits_semantic(
            crop_logits,
            origins,
            img_sizes
        )

    return logits

In [9]:
import glob
import numpy as np
import torch
from PIL import Image

BASE_PATH = "/content/drive/MyDrive/Validation_Dataset"

test_img_paths = sorted(glob.glob(f"{BASE_PATH}/RoadAnomaly21/images/*"))
print("Numero immagini test:", len(test_img_paths))

test_path = test_img_paths[0]
print("Test image:", test_path)

# 1. Lettura immagine (Viene mantenuta a 512x1024 come da input_transform)
img_pil = input_transform(Image.open(test_path).convert("RGB"))

img_tensor = torch.from_numpy(
    np.array(img_pil)
).permute(2, 0, 1)

print("img_tensor shape (Input):", img_tensor.shape)
print("img_tensor dtype:", img_tensor.dtype)

# ----------------------------------------------------------------------
# ASSEGNAZIONE AUTOMATICA DELLA GEOMETRIA (Previene l'AssertionError)
# ----------------------------------------------------------------------
if USE_COCO:
    target_img_size = (640, 640)
    model.img_size = (640, 640)
    model.window_size = 640
else:
    target_img_size = (1024, 1024)
    model.img_size = (1024, 1024)
    model.window_size = 1024

print(f"\nUso configurazione per MODELLO: {MODEL_NAME.upper()}")
print(f"  -> Finestra interna (window_size): {model.window_size}")

# 2. Esecuzione dell'inferenza a finestre
logits_list = get_eomt_logits(
    model=model,
    img_tensor=img_tensor,
    target_size=target_img_size
)

print("\n--- RISULTATO INFERENZA ---")
print("Tipo di output:", type(logits_list))
print("Numero di layer nella lista:", len(logits_list))
# Con COCO stamperà torch.Size([133, 512, 1024])
print("Forma finale dei logit ricostruiti:", logits_list[0].shape)

Numero immagini test: 10
Test image: /content/drive/MyDrive/Validation_Dataset/RoadAnomaly21/images/0.png
img_tensor shape (Input): torch.Size([3, 512, 1024])
img_tensor dtype: torch.uint8

Uso configurazione per MODELLO: CITYSCAPES
  -> Finestra interna (window_size): 1024

--- RISULTATO INFERENZA ---
Tipo di output: <class 'list'>
Numero di layer nella lista: 1
Forma finale dei logit ricostruiti: torch.Size([19, 512, 1024])


In [10]:
import glob
import numpy as np
import torch
from PIL import Image

BASE_PATH = "/content/drive/MyDrive/Validation_Dataset"

test_img_paths = sorted(glob.glob(f"{BASE_PATH}/RoadAnomaly21/images/*"))
print("Numero immagini test:", len(test_img_paths))

test_path = test_img_paths[0]
print("Test image:", test_path)

# 1. Lettura immagine (512x1024)
img_pil = input_transform(Image.open(test_path).convert("RGB"))

img_tensor = torch.from_numpy(
    np.array(img_pil)
).permute(2, 0, 1)

print("img_tensor shape (Input):", img_tensor.shape)
print("img_tensor dtype:", img_tensor.dtype)

# ----------------------------------------------------------------------
# CONFIGURAZIONE DINAMICA (Usa i parametri impostati dal caricamento)
# ----------------------------------------------------------------------
print(f"\nUso configurazione per MODELLO: {MODEL_NAME.upper()}")
print(f"  -> Finestra interna (window_size): {model.window_size}")

# 2. Esecuzione dell'inferenza a finestre senza sovrascrivere nulla
logits_list = get_eomt_logits(
    model=model,
    img_tensor=img_tensor,
    target_size=target_img_size # Sarà (640, 640) per COCO o (1024, 1024) per Cityscapes
)

print("\n--- RISULTATO INFERENZA ---")
print("Tipo di output:", type(logits_list))
print("Numero di layer nella lista:", len(logits_list))
# Se stai usando COCO stamperà torch.Size([133, 512, 1024])
# Se stai usando Cityscapes stamperà torch.Size([19, 512, 1024])
print("Forma finale dei logit ricostruiti:", logits_list[0].shape)

Numero immagini test: 10
Test image: /content/drive/MyDrive/Validation_Dataset/RoadAnomaly21/images/0.png
img_tensor shape (Input): torch.Size([3, 512, 1024])
img_tensor dtype: torch.uint8

Uso configurazione per MODELLO: CITYSCAPES
  -> Finestra interna (window_size): 1024

--- RISULTATO INFERENZA ---
Tipo di output: <class 'list'>
Numero di layer nella lista: 1
Forma finale dei logit ricostruiti: torch.Size([19, 512, 1024])


In [11]:
import shutil
import os

SAVED_LOGITS_ROOT = "/content/saved_logits_step8"

if os.path.exists(SAVED_LOGITS_ROOT):
    shutil.rmtree(SAVED_LOGITS_ROOT)
    print(f"Cartella cancellata: {SAVED_LOGITS_ROOT}")

os.makedirs(SAVED_LOGITS_ROOT, exist_ok=True)
print(f"Cartella ricreata: {SAVED_LOGITS_ROOT}")

Cartella ricreata: /content/saved_logits_step8


In [12]:
import glob
import os
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

BASE_PATH = "/content/drive/MyDrive/Validation_Dataset"
SAVED_LOGITS_ROOT = "/content/saved_logits_step8"

# ----------------------------------------------------------------------
# GEOMETRIA DINAMICA: Rileva automaticamente il modello attivo
# ----------------------------------------------------------------------
if USE_COCO:
    target_img_size = (640, 640)
    model.img_size = (640, 640)
    model.window_size = 640
    num_classes_calc = 133
else:
    target_img_size = (1024, 1024)
    model.img_size = (1024, 1024)
    model.window_size = 1024
    num_classes_calc = 19

datasets = {
    "SMIYC RA-21": "RoadAnomaly21",
    "SMIYC RO-21": "RoadObsticle21",
    "FS L&F": "FS_LostFound_full",
    "FS Static": "fs_static",
    "Road Anomaly": "RoadAnomaly"
}

all_results = {}

for dataset_name, dataset_folder in datasets.items():

    print("\n" + "=" * 80)
    print(f"MODELLO: {MODEL_NAME.upper()}")
    print(f"DATASET: {dataset_name} ({dataset_folder})")
    print("=" * 80)

    ood_gts_list = []

    anomaly_scores_dict = {
        "max_logit": [],
        "msp": [],
        "entropy": [],
        "rba": []
    }

    cartella_logits = os.path.join(
        SAVED_LOGITS_ROOT,
        f"saved_logits_{MODEL_NAME}_{dataset_folder}"
    )
    os.makedirs(cartella_logits, exist_ok=True)
    print(f"Logits salvati in: {cartella_logits}")

    image_paths = sorted(glob.glob(f"{BASE_PATH}/{dataset_folder}/images/*"))
    print(f"Numero immagini trovate: {len(image_paths)}")

    for idx, path in enumerate(image_paths):

        if idx % 10 == 0:
            print(f"  Processing image {idx + 1}/{len(image_paths)}")

        # -------------------------
        # Lettura immagine
        # -------------------------
        img_pil = input_transform(Image.open(path).convert("RGB"))
        img_tensor = torch.from_numpy(np.array(img_pil)).permute(2, 0, 1)

        # -------------------------
        # Inferenza EoMT a finestre
        # -------------------------
        logits_list = get_eomt_logits(
            model=model,
            img_tensor=img_tensor,
            target_size=target_img_size
        )
        logits = logits_list[0].unsqueeze(0)  # [1, C, 512, 1024]

        # -------------------------
        # Salvataggio logit per Step 9 (Temperature Scaling)
        # -------------------------
        nome_base = os.path.basename(path).replace(".jpg", "").replace(".png", "").replace(".webp", "")
        save_path_logits = os.path.join(cartella_logits, f"{nome_base}.pt")
        torch.save(logits.detach().cpu(), save_path_logits)

        # -------------------------
        # Calcolo anomaly scores
        # -------------------------
        probs = F.softmax(logits, dim=1)
        probs_no_batch = probs.squeeze(0)
        logits_no_batch = logits.squeeze(0)

        # MSP
        score_msp = 1.0 - torch.max(probs_no_batch, dim=0)[0]
        score_msp = score_msp.detach().cpu().numpy()

        # MaxLogit
        score_maxLogit = 1.0 - torch.max(logits_no_batch, dim=0)[0]
        score_maxLogit = score_maxLogit.detach().cpu().numpy()

        # MaxEntropy (Normalizzato dinamicamente sul numero esatto di classi)
        entropy = -torch.sum(probs_no_batch * torch.log(probs_no_batch + 1e-10), dim=0)
        #score_entropy = (entropy / np.log(num_classes_calc)).detach().cpu().numpy()
        score_entropy = entropy.data.detach().cpu().numpy()

        # RbA
        score_rba = -torch.sum(torch.tanh(logits_no_batch), dim=0)
        score_rba = score_rba.detach().cpu().numpy()

        # -------------------------
        # Lettura Ground Truth
        # -------------------------
        pathGT = path.replace("images", "labels_masks")
        if dataset_folder == "RoadObsticle21":
            pathGT = pathGT.replace("webp", "png")
        if dataset_folder in ["fs_static", "RoadAnomaly", "RoadAnomaly21", "FS_LostFound_full"]:
            pathGT = pathGT.replace("jpg", "png")

        if not os.path.exists(pathGT):
            print(f"GT non trovata: {pathGT}")
            continue

        mask = Image.open(pathGT)
        mask = target_transform(mask)
        ood_gts = np.array(mask)

        # Binarizzazione classe anomala per RoadAnomaly
        if "RoadAnomaly" in pathGT:
            ood_gts = np.where(ood_gts == 2, 1, ood_gts)

        # Esclusione corretta delle ignore regions (255) dal calcolo
        # Nota: Non li convertiamo a 0, lasciando che rimangano 255.
        # Nella cella delle metriche verranno automaticamente ignorati
        # sia da ood_mask (==1) sia da ind_mask (==0), ottimizzando l'accuratezza.

        # Filtro protettivo per immagini valide
        if 1 not in np.unique(ood_gts):
            continue

        # -------------------------
        # Stoccaggio vettori linearizzati
        # -------------------------
        anomaly_scores_dict["max_logit"].append(score_maxLogit.ravel())
        anomaly_scores_dict["msp"].append(score_msp.ravel())
        anomaly_scores_dict["entropy"].append(score_entropy.ravel())
        anomaly_scores_dict["rba"].append(score_rba.ravel())
        ood_gts_list.append(ood_gts.ravel())

        del logits, probs, score_msp, score_maxLogit, score_entropy, score_rba, ood_gts
        torch.cuda.empty_cache()

    if len(ood_gts_list) == 0:
        print(f"Nessuna immagine valida trovata per {dataset_name}")
        continue

    # Compattazione dei vettori nel dizionario globale
    all_results[dataset_name] = {
        "gt": np.concatenate(ood_gts_list),
        "max_logit": np.concatenate(anomaly_scores_dict["max_logit"]),
        "msp": np.concatenate(anomaly_scores_dict["msp"]),
        "entropy": np.concatenate(anomaly_scores_dict["entropy"]),
        "rba": np.concatenate(anomaly_scores_dict["rba"])
    }

    print(f"Immagini valide usate: {len(ood_gts_list)}")

print("\nDataset salvati correttamente in all_results:")
print(list(all_results.keys()))


MODELLO: CITYSCAPES
DATASET: SMIYC RA-21 (RoadAnomaly21)
Logits salvati in: /content/saved_logits_step8/saved_logits_cityscapes_RoadAnomaly21
Numero immagini trovate: 10
  Processing image 1/10
Immagini valide usate: 10

MODELLO: CITYSCAPES
DATASET: SMIYC RO-21 (RoadObsticle21)
Logits salvati in: /content/saved_logits_step8/saved_logits_cityscapes_RoadObsticle21
Numero immagini trovate: 30
  Processing image 1/30
  Processing image 11/30
  Processing image 21/30
Immagini valide usate: 30

MODELLO: CITYSCAPES
DATASET: FS L&F (FS_LostFound_full)
Logits salvati in: /content/saved_logits_step8/saved_logits_cityscapes_FS_LostFound_full
Numero immagini trovate: 100
  Processing image 1/100
  Processing image 11/100
  Processing image 21/100
  Processing image 31/100
  Processing image 41/100
  Processing image 51/100
  Processing image 61/100
  Processing image 71/100
  Processing image 81/100
  Processing image 91/100
Immagini valide usate: 99

MODELLO: CITYSCAPES
DATASET: FS Static (fs_st

In [13]:
from sklearn.metrics import average_precision_score
import numpy as np

final_table = {}

for dataset_name, data in all_results.items():

    print("\n" + "=" * 80)
    print(f"RISULTATI DATASET: {dataset_name}")
    print("=" * 80)

    ood_gts_all = data["gt"]

    ood_mask = ood_gts_all == 1
    ind_mask = ood_gts_all == 0

    # CORREZIONE: Aggiungiamo la RbA al dizionario dei punteggi da valutare
    concatenated_scores = {
        "MSP": data["msp"],
        "MaxLogit": data["max_logit"],
        "MaxEntropy": data["entropy"],
        "RbA": data["rba"]  # <--- Inserito qui per completare la traccia!
    }

    final_table[dataset_name] = {}

    for method_name, current_score in concatenated_scores.items():

        ood_out = current_score[ood_mask]
        ind_out = current_score[ind_mask]

        val_out = np.concatenate((ind_out, ood_out))

        val_label = np.concatenate((
            np.zeros(len(ind_out)),
            np.ones(len(ood_out))
        ))

        auprc = average_precision_score(val_label, val_out)
        fpr = fpr_at_95_tpr(val_out, val_label)

        final_table[dataset_name][method_name] = {
            "AUPRC": auprc * 100,
            "FPR95": fpr * 100
        }

        print(method_name)
        print(f"  AUPRC: {auprc * 100:.2f}%")
        print(f"  FPR95: {fpr * 100:.2f}%")
        print("-" * 40)


RISULTATI DATASET: SMIYC RA-21
MSP
  AUPRC: 69.59%
  FPR95: 14.10%
----------------------------------------
MaxLogit
  AUPRC: 69.44%
  FPR95: 14.38%
----------------------------------------
MaxEntropy
  AUPRC: 69.83%
  FPR95: 14.35%
----------------------------------------
RbA
  AUPRC: 65.52%
  FPR95: 95.60%
----------------------------------------

RISULTATI DATASET: SMIYC RO-21
MSP
  AUPRC: 84.27%
  FPR95: 4.49%
----------------------------------------
MaxLogit
  AUPRC: 84.11%
  FPR95: 4.61%
----------------------------------------
MaxEntropy
  AUPRC: 84.32%
  FPR95: 4.62%
----------------------------------------
RbA
  AUPRC: 80.75%
  FPR95: 99.91%
----------------------------------------

RISULTATI DATASET: FS L&F
MSP
  AUPRC: 16.85%
  FPR95: 10.17%
----------------------------------------
MaxLogit
  AUPRC: 16.79%
  FPR95: 9.94%
----------------------------------------
MaxEntropy
  AUPRC: 19.22%
  FPR95: 9.93%
----------------------------------------
RbA
  AUPRC: 16.51%
  FPR95: 5.

#Temperature Scaling

In [14]:
import os
import glob
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import average_precision_score

temperature_da_testare = [0.01, 0.05, 0.1, 0.2, 0.5, 0.75, 1.0, 1.1, 1.5, 2.0, 5.0, 10.0, 100.0]

temperature_results = {}

for dataset_name, dataset_folder in datasets.items():

    print("\n" + "=" * 80)
    print(f"TEMPERATURE SCALING - MODELLO: {MODEL_NAME.upper()} - DATASET: {dataset_name}")
    print("=" * 80)

    cartella_logits = os.path.join(
        SAVED_LOGITS_ROOT,
        f"saved_logits_{MODEL_NAME}_{dataset_folder}"
    )

    if not os.path.exists(cartella_logits):
        print(f"Cartella logits non trovata: {cartella_logits}")
        continue

    logit_files = sorted(glob.glob(os.path.join(cartella_logits, "*.pt")))

    if len(logit_files) == 0:
        print(f"Nessun file .pt trovato in: {cartella_logits}")
        continue

    print(f"Numero logits trovati: {len(logit_files)}")

    temperature_results[dataset_name] = {}

    for T in temperature_da_testare:

        gts_T = []
        score_msp_T = []

        for logit_path in logit_files:

            # I logits salvati nel ciclo principale sono direttamente tensori [1, C, H, W]
            logits = torch.load(logit_path, map_location="cpu").float()

            # Temperature scaling
            probs_T = F.softmax(logits / T, dim=1)

            # MSP anomaly score: più alto = più anomalo
            anomaly_msp = 1.0 - torch.max(probs_T, dim=1)[0]
            anomaly_msp = anomaly_msp.squeeze(0).numpy()

            # Ricostruisco il path della GT a partire dal nome del file logits
            nome_base = os.path.basename(logit_path).replace(".pt", "")

            possible_gt_paths = [
                os.path.join(BASE_PATH, dataset_folder, "labels_masks", f"{nome_base}.png"),
                os.path.join(BASE_PATH, dataset_folder, "labels_masks", f"{nome_base}.jpg"),
                os.path.join(BASE_PATH, dataset_folder, "labels_masks", f"{nome_base}.webp"),
            ]

            pathGT = None
            for candidate in possible_gt_paths:
                if os.path.exists(candidate):
                    pathGT = candidate
                    break

            if pathGT is None:
                print(f"GT non trovata per: {nome_base}")
                continue

            mask = Image.open(pathGT)
            mask = target_transform(mask)
            ood_gts = np.array(mask)

            # Stessa conversione del ciclo principale
            if "RoadAnomaly" in pathGT:
                ood_gts = np.where(ood_gts == 2, 1, ood_gts)

            # Salto immagini senza anomalie
            if 1 not in np.unique(ood_gts):
                continue

            # Tengo solo pixel validi: 0 = ID, 1 = OOD
            # Escludo automaticamente ignore regions 255
            valid_mask = (ood_gts == 0) | (ood_gts == 1)

            gts_T.append(ood_gts[valid_mask].ravel())
            score_msp_T.append(anomaly_msp[valid_mask].ravel())

        if len(gts_T) == 0:
            print(f"Warning: nessuna GT valida per T={T}")
            continue

        val_gts = np.concatenate(gts_T)
        val_msp = np.concatenate(score_msp_T)

        auprc_msp = average_precision_score(val_gts, val_msp)
        fpr_msp = fpr_at_95_tpr(val_msp, val_gts)

        temperature_results[dataset_name][T] = {
            "AUPRC": auprc_msp * 100,
            "FPR95": fpr_msp * 100
        }

        print(f"T = {T}")
        print(f"  AUPRC MSP: {auprc_msp * 100:.2f}%")
        print(f"  FPR95 MSP: {fpr_msp * 100:.2f}%")
        print("-" * 40)


TEMPERATURE SCALING - MODELLO: CITYSCAPES - DATASET: SMIYC RA-21
Numero logits trovati: 10
T = 0.01
  AUPRC MSP: 49.16%
  FPR95 MSP: 90.01%
----------------------------------------
T = 0.05
  AUPRC MSP: 71.74%
  FPR95 MSP: 9.10%
----------------------------------------
T = 0.1
  AUPRC MSP: 69.07%
  FPR95 MSP: 13.98%
----------------------------------------
T = 0.2
  AUPRC MSP: 69.24%
  FPR95 MSP: 14.07%
----------------------------------------
T = 0.5
  AUPRC MSP: 69.45%
  FPR95 MSP: 14.10%
----------------------------------------
T = 0.75
  AUPRC MSP: 69.54%
  FPR95 MSP: 14.10%
----------------------------------------
T = 1.0
  AUPRC MSP: 69.60%
  FPR95 MSP: 14.10%
----------------------------------------
T = 1.1
  AUPRC MSP: 69.62%
  FPR95 MSP: 14.10%
----------------------------------------
T = 1.5
  AUPRC MSP: 69.71%
  FPR95 MSP: 14.10%
----------------------------------------
T = 2.0
  AUPRC MSP: 69.77%
  FPR95 MSP: 14.10%
----------------------------------------
T = 5.0
  AUPRC 